In [1]:
import sys
from pathlib import Path

# Asegura que el path src/ esté disponible
ROOT = Path("/home/jovyan/work/")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.spark import get_spark
from src.io import read_parquet
from src.config import Paths
# Módulos con la lógica de profiling y utilidades
from src.profiling import (
    calculate_spark_stats, 
    report_categorical_freq, 
    compute_corr_matrix_and_report,
    show_head_and_schema # Ejemplo de uso de otra función de profiling.py
)
from src.utils import print_header # Para el formato de títulos
import src.config as cfg


In [2]:
spark = get_spark("PerfiladoPipeline")

# Leer el DF combinado (generado por Separación_datasets.ipynb)
df_combined = read_parquet(spark, cfg.COMBINED_PATH)

print_header(f"DF Combinado: {df_combined.count()} filas")
show_head_and_schema(df_combined, "DF Combinado")


DF Combinado: 1688934 filas

DF Combinado - head(5)
+------------+----------------------+-----------------+------+------------+--------------+-----------+---------+--------------+--------------+------------+-----------------+-------------+--------------+-----------+-----------------+-----------+------------------+------------------+---------------------+------------------+----------+--------------+----------+--------------------------+---------------------+------------------------+------------------------+------------------------+---------------------------+---------------------------+---------------------------+-----------------------+-----------------------+-----------------------+----------------------+----------------------+-------------------+---------------------+-----------------+-------------------+----------------+----------+--------------------+-----------------+------------------------+--------------------+------------------------+--------------------------+----------------

In [3]:
print_header("1. Estadísticas Descriptivas Detalladas")
# Llama a la función modular (usa la versión corregida de profiling.py)
stats_df = calculate_spark_stats(df_combined, "DF Combinado") 
stats_df.show(truncate=False)


1. Estadísticas Descriptivas Detalladas
+-------------------+---------------------------+--------------------+------------------+--------------------+------------------+--------------------+------------------+
|CoefVar            |Columna                    |DesvStd             |Max               |Media               |Mediana           |Min                 |Moda              |
+-------------------+---------------------------+--------------------+------------------+--------------------+------------------+--------------------+------------------+
|3.602323840323574  |NON_COMPLIANT_CONTRACT     |0.2577372535225535  |1.0               |0.07154749682344011 |0.0               |0.0                 |0.0               |
|0.5454289221487342 |TOTAL_INCOME               |1179.4333366404876  |43200.0           |2162.3960313546877  |1890.0            |324.0               |1620.0            |
|0.6037833441321204 |AMOUNT_PRODUCT             |4834.588594433785   |48486.195         |8007.157934081527   

In [4]:
report_categorical_freq(df_combined, "DF Combinado", max_categories=20)


DF Combinado - Frecuencias Categóricas (<= 20)

===== Frecuencia de NAME_PRODUCT_TYPE =====
+-----------------+-------+
|NAME_PRODUCT_TYPE|count  |
+-----------------+-------+
|PRODUCT 1        |1660120|
|PRODUCT 2        |28814  |
+-----------------+-------+


===== Frecuencia de GENDER =====
+------+-------+
|GENDER|count  |
+------+-------+
|F     |1134067|
|M     |554867 |
+------+-------+


===== Frecuencia de EDUCATION =====
+---------------------+-------+
|EDUCATION            |count  |
+---------------------+-------+
|Secondary            |1252636|
|NULL                 |373289 |
|Incomplete University|44756  |
|Primary School       |16868  |
|Master/PhD           |1385   |
+---------------------+-------+


===== Frecuencia de MARITAL_STATUS =====
+--------------+-------+
|MARITAL_STATUS|count  |
+--------------+-------+
|Married       |1311050|
|Single        |377884 |
+--------------+-------+


===== Frecuencia de HOME_SITUATION =====
+-----------------------+-------+
|HOME_

In [5]:
print_header("3. Matriz de Correlación")
# La función calcula la matriz y reporta los pares con alta correlación
corr_df = compute_corr_matrix_and_report(df_combined, "DF Combinado", threshold=0.4)


3. Matriz de Correlación

Correlaciones |r| >= 0.4 en DF Combinado
+---------------------------+---------------------------+-------------------+-------------------+
|Variable_A                 |Variable_B                 |Correlacion        |Abs_Correlacion    |
+---------------------------+---------------------------+-------------------+-------------------+
|LOAN_APPLICATION_AMOUNT_SUM|LOAN_CREDIT_GRANTED_SUM    |0.9935752940083173 |0.9935752940083173 |
|LOAN_APPLICATION_AMOUNT_MAX|LOAN_CREDIT_GRANTED_MAX    |0.9844036710607148 |0.9844036710607148 |
|LOAN_ANNUITY_PAYMENT_SUM   |LOAN_CREDIT_GRANTED_SUM    |0.9155541344858282 |0.9155541344858282 |
|LOAN_ANNUITY_PAYMENT_SUM   |LOAN_APPLICATION_AMOUNT_SUM|0.9096911225546751 |0.9096911225546751 |
|LOAN_APPLICATION_AMOUNT_MIN|LOAN_CREDIT_GRANTED_MIN    |0.8790645419324384 |0.8790645419324384 |
|LOAN_ANNUITY_PAYMENT_MIN   |LOAN_CREDIT_GRANTED_MIN    |0.8435804115286841 |0.8435804115286841 |
|CREDIT_CARD_DRAWINGS_ATM   |CREDIT_CARD_DRAWINGS 

Al pasarlo a PySpark hay un error del orden de 0.00x a 0.02–0.03 en muchos coeficientes.

Eso no es enorme en términos de correlación:

- No cambia el signo (positiva sigue siendo positiva, negativa sigue siendo negativa).

- La interpretación “fuerte / débil / casi 0” suele ser la misma.